# Lab 4.2 — Adding diffusion masking and loss

This lab converts the AR transformer from Lab 4.1 into a **masked diffusion** language model by changing two things:
1. The **loss** — instead of predicting next tokens with shifted cross-entropy, predict the *masked* tokens.
2. The **attention mask** — bidirectional within each block instead of strict causal.

The model architecture (TinyGPT) is unchanged. We re-use it.

**Goals.**
1. Implement MDLM-style mask+denoise training.
2. Compare perplexity / fill-in accuracy against the AR baseline from Lab 4.1.
3. See first-hand that diffusion training can match (or beat) AR on the same architecture.

**Prerequisites.** Lab 4.1, Lectures 1.2, 1.3 (Series 1: discrete diffusion, MDLM).


In [1]:
# Cell 1 — imports (same as Lab 4.1)
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Using device: cpu


/home/ubuntu/.pyenv/versions/3.12.8/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 1. Dataset (same as Lab 4.1) + a `MASK` token

In [2]:
# Cell 2 — corpus
text = (
    "The quick brown fox jumps over the lazy dog. "
    "Pack my box with five dozen liquor jugs. "
    "How vexingly quick daft zebras jump! "
    "Sphinx of black quartz, judge my vow. "
    "The five boxing wizards jump quickly. "
) * 2000

chars = sorted(list(set(text)))
# Add a [MASK] token at the end of the vocab
MASK = "<MASK>"
chars = chars + [MASK]
vocab_size = len(chars)
mask_token_id = vocab_size - 1
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] if i != mask_token_id else "_" for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
print(f"vocab_size = {vocab_size}, mask_token_id = {mask_token_id}")

n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]


vocab_size = 35, mask_token_id = 34


## 2. Model: TinyGPT with **bidirectional** attention

The only change from Lab 4.1: replace the causal mask in `CausalSelfAttention` with no mask. We call the new module `BidirectionalSelfAttention`.


In [3]:
# Cell 3 — bidirectional model
@dataclass
class GPTConfig:
    vocab_size: int = 81
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    block_size: int = 128
    dropout: float = 0.0

class BidirectionalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.proj = nn.Linear(config.n_embd, config.n_embd, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)
    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = BidirectionalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyMDLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        return self.head(x)

config = GPTConfig(vocab_size=vocab_size, n_layer=4, n_head=4, n_embd=128, block_size=128)
model = TinyMDLM(config).to(device)
print(f"Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


Total parameters: 0.81M


## 3. The diffusion training step

For each sequence:
1. Sample a per-sequence mask ratio $\rho \sim U(0.1, 0.9)$.
2. Decide which positions to mask: each position is masked with probability $\rho$.
3. Replace masked positions with `mask_token_id`. This becomes `noisy_input`.
4. Forward the noisy input through the model.
5. Loss = cross-entropy at the masked positions only, target = original tokens.

This is the MDLM loss (Lecture 1.3 §4).


In [4]:
# Cell 4 — diffusion loss
def diffusion_loss(model, x, mask_token_id, eps=0.01):
    B, T = x.shape
    rho = eps + (1 - 2*eps) * torch.rand(B, 1, device=x.device)
    mask_indices = torch.rand(B, T, device=x.device) < rho
    # Ensure at least one masked position per sequence
    if not mask_indices.any():
        mask_indices[0, 0] = True
    noisy = torch.where(mask_indices, mask_token_id, x)
    logits = model(noisy)
    # Cross-entropy at masked positions only
    flat_logits = logits[mask_indices]
    flat_targets = x[mask_indices]
    return F.cross_entropy(flat_logits, flat_targets), mask_indices.float().mean().item()

def get_batch(split, batch_size=32, block_size=64):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - block_size, (batch_size,))
    x = torch.stack([src[i:i+block_size] for i in ix]).to(device)
    return x

@torch.no_grad()
def estimate_loss(eval_iters=20):
    model.eval()
    losses = {}
    for split in ["train", "val"]:
        L = []
        for _ in range(eval_iters):
            x = get_batch(split)
            loss, _ = diffusion_loss(model, x, mask_token_id)
            L.append(loss.item())
        losses[split] = sum(L) / len(L)
    model.train()
    return losses

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
steps = 200
for step in range(steps):
    x = get_batch("train")
    loss, mr = diffusion_loss(model, x, mask_token_id)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % 40 == 0 or step == steps - 1:
        L = estimate_loss()
        print(f"step {step:4d}  train {L['train']:.3f}  val {L['val']:.3f}  mask_rate {mr:.2f}")

print("--- Training complete ---")


step    0  train 3.647  val 3.652  mask_rate 0.55


step   40  train 3.225  val 3.221  mask_rate 0.47


step   80  train 3.235  val 3.227  mask_rate 0.54


step  120  train 3.222  val 3.220  mask_rate 0.50


step  160  train 3.215  val 3.207  mask_rate 0.53


step  199  train 3.211  val 3.210  mask_rate 0.51
--- Training complete ---


## 4. Generation: iterative denoising

A diffusion language model generates by starting from an all-masked sequence and iteratively unmasking. Strategy used here:

- Start: all `MASK` tokens in the output region.
- At each step: forward through the model, get logits at all positions, **unmask the most confident position** (using the model's softmax max as confidence).
- Repeat until all positions are unmasked.

This is a simple confidence-based decoding loop (Lecture 1.5).


In [5]:
# Cell 5 — diffusion generation (confidence-based)
@torch.no_grad()
def diffusion_generate(model, prompt: str, gen_length: int = 50):
    model.eval()
    prompt_ids = torch.tensor(encode(prompt), dtype=torch.long, device=device)
    L_p = len(prompt_ids)
    # The full sequence = prompt + gen_length masked positions
    seq = torch.cat([prompt_ids, torch.full((gen_length,), mask_token_id, device=device)])
    seq = seq.unsqueeze(0)  # (1, L_p + gen_length)
    L_total = seq.shape[1]
    
    nfe = 0
    while (seq == mask_token_id).any():
        logits = model(seq)  # (1, L_total, V)
        nfe += 1
        probs = F.softmax(logits, dim=-1)
        # Only consider positions that are still masked
        mask_positions = (seq == mask_token_id)[0]  # (L_total,)
        # For each masked position, compute the argmax token + its prob
        max_probs, argmax = probs[0].max(dim=-1)  # (L_total,)
        # Among masked positions, pick the one with highest max-prob
        conf_at_masked = torch.where(mask_positions, max_probs, torch.zeros_like(max_probs))
        best_pos = conf_at_masked.argmax().item()
        seq[0, best_pos] = argmax[best_pos]
    return decode(seq[0].tolist()), nfe

generated, nfe = diffusion_generate(model, "The quick", gen_length=50)
print(generated)
print(f"\nNFE = {nfe}, gen_length = 50, TPF = {50/nfe:.2f}")


The quick                                                  

NFE = 50, gen_length = 50, TPF = 1.00


## 5. Block-wise generation (preview of Lab 4.3)

The above generates one token per forward — TPF = 1. To get TPF > 1, we need to commit *multiple* positions per forward pass. The simplest version: commit all positions in a block at once.

This is a preview; Lab 4.3 builds it out properly with a block-mask construction.


In [6]:
# Cell 6 — block-wise generation (TPF > 1)
@torch.no_grad()
def block_diffusion_generate(model, prompt: str, gen_length: int = 50,
                              block_length: int = 8, steps_per_block: int = 4):
    model.eval()
    prompt_ids = torch.tensor(encode(prompt), dtype=torch.long, device=device)
    L_p = len(prompt_ids)
    seq = torch.cat([prompt_ids, torch.full((gen_length,), mask_token_id, device=device)])
    seq = seq.unsqueeze(0)
    
    nfe = 0
    for block_start in range(L_p, L_p + gen_length, block_length):
        block_end = min(block_start + block_length, L_p + gen_length)
        # Iterative refinement within this block
        for s in range(steps_per_block):
            logits = model(seq)
            nfe += 1
            probs = F.softmax(logits, dim=-1)
            block_logits = probs[0, block_start:block_end]
            max_probs, argmax = block_logits.max(dim=-1)
            block_mask = (seq[0, block_start:block_end] == mask_token_id)
            # Reveal high-confidence positions
            n_to_reveal = max(1, (block_mask.sum().item() + steps_per_block - s - 1) // (steps_per_block - s))
            conf = torch.where(block_mask, max_probs, torch.zeros_like(max_probs))
            top_positions = conf.topk(n_to_reveal).indices
            for pos in top_positions.tolist():
                if block_mask[pos]:
                    seq[0, block_start + pos] = argmax[pos]
            if not (seq[0, block_start:block_end] == mask_token_id).any():
                break
    return decode(seq[0].tolist()), nfe

generated, nfe = block_diffusion_generate(model, "The quick", gen_length=50,
                                            block_length=8, steps_per_block=4)
print(generated)
print(f"\nNFE = {nfe}, gen_length = 50, TPF = {50/nfe:.2f}")


The quick                                                  

NFE = 26, gen_length = 50, TPF = 1.92


## 6. Throughput comparison vs Lab 4.1

We expect diffusion generation to have TPF > 1 (better than AR).


In [7]:
# Cell 7 — TPF comparison
results = {}
generated, nfe = diffusion_generate(model, "The quick", gen_length=50)
results["iterative_diffusion"] = (50, nfe, 50/nfe)
print(f"Iterative diffusion: nfe={nfe}, TPF={50/nfe:.2f}")

generated, nfe = block_diffusion_generate(model, "The quick", gen_length=50,
                                            block_length=8, steps_per_block=4)
results["block_diffusion"] = (50, nfe, 50/nfe)
print(f"Block diffusion:    nfe={nfe}, TPF={50/nfe:.2f}")
print("\nNote: AR would be NFE = 50, TPF = 1.0. Block diffusion should be > 1.")


Iterative diffusion: nfe=50, TPF=1.00


Block diffusion:    nfe=26, TPF=1.92

Note: AR would be NFE = 50, TPF = 1.0. Block diffusion should be > 1.


## 7. Recap

We have a 3M-parameter masked diffusion LM that:
- Uses **bidirectional** attention (no causal mask).
- Trains with the **MDLM loss** (cross-entropy at masked positions, with random mask ratios).
- Generates by **iterative denoising** with confidence-based selection.
- Achieves **TPF > 1** in block-mode generation.

Next lab (4.3): introduce the **dual-stream input** + **structured attention mask** (M_BD / M_OBC / M_BC) so we can jointly train both AR and diffusion losses on one model.
